
RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf/Shakespeare/")

Found 6 PDF files to process

Processing: as-you-like-it_PDF_FolgerShakespeare.pdf
  ✓ Loaded 103 pages

Processing: a-midsummer-nights-dream_PDF_FolgerShakespeare.pdf
  ✓ Loaded 87 pages

Processing: alls-well-that-ends-well_PDF_FolgerShakespeare.pdf
  ✓ Loaded 110 pages

Processing: cymbeline_PDF_FolgerShakespeare.pdf
  ✓ Loaded 132 pages

Processing: antony-and-cleopatra_PDF_FolgerShakespeare.pdf
  ✓ Loaded 140 pages

Processing: coriolanus_PDF_FolgerShakespeare.pdf
  ✓ Loaded 142 pages

Total documents loaded: 714


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m83', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.106 Safari/537.36', 'creationdate': '2020-06-24T23:59:42+00:00', 'moddate': '2020-06-24T23:59:42+00:00', 'source': '../data/pdf/Shakespeare/as-you-like-it_PDF_FolgerShakespeare.pdf', 'total_pages': 103, 'page': 0, 'page_label': '1', 'source_file': 'as-you-like-it_PDF_FolgerShakespeare.pdf', 'file_type': 'pdf'}, page_content='Get even more from the Folger\nYou can get your own copy of this text to keep. Purchase a full copy\nto get the text, plus explanatory notes, illustrations, and more.\n Buy a copy\nFolger Shakespeare Library\nhttps://shakespeare.folger.edu/'),
 Document(metadata={'producer': 'Skia/PDF m83', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.106 Safari/537.36', 'creationdate': '2020-06-24T23:59:42+00:00', 'moddate': '2020-06-24T23:59:42+00:0

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 714 documents into 1544 chunks

Example chunk:
Content: Get even more from the Folger
You can get your own copy of this text to keep. Purchase a full copy
to get the text, plus explanatory notes, illustrations, and more.
 Buy a copy
Folger Shakespeare Libr...
Metadata: {'producer': 'Skia/PDF m83', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.106 Safari/537.36', 'creationdate': '2020-06-24T23:59:42+00:00', 'moddate': '2020-06-24T23:59:42+00:00', 'source': '../data/pdf/Shakespeare/as-you-like-it_PDF_FolgerShakespeare.pdf', 'total_pages': 103, 'page': 0, 'page_label': '1', 'source_file': 'as-you-like-it_PDF_FolgerShakespeare.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m83', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.106 Safari/537.36', 'creationdate': '2020-06-24T23:59:42+00:00', 'moddate': '2020-06-24T23:59:42+00:00', 'source': '../data/pdf/Shakespeare/as-you-like-it_PDF_FolgerShakespeare.pdf', 'total_pages': 103, 'page': 0, 'page_label': '1', 'source_file': 'as-you-like-it_PDF_FolgerShakespeare.pdf', 'file_type': 'pdf'}, page_content='Get even more from the Folger\nYou can get your own copy of this text to keep. Purchase a full copy\nto get the text, plus explanatory notes, illustrations, and more.\n Buy a copy\nFolger Shakespeare Library\nhttps://shakespeare.folger.edu/'),
 Document(metadata={'producer': 'Skia/PDF m83', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.106 Safari/537.36', 'creationdate': '2020-06-24T23:59:42+00:00', 'moddate': '2020-06-24T23:59:42+00:0

embedding And vectorStoreDB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


VectorStore

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1544


In [9]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 1544 texts...


Batches: 100%|██████████| 49/49 [00:02<00:00, 16.93it/s]


Generated embeddings with shape: (1544, 384)
Adding 1544 documents to vector store...
Successfully added 1544 documents to vector store
Total documents in collection: 1544


Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever

In [19]:
rag_retriever.retrieve("""Then it will be
FTLN 1308 120
the earliest fruit i’ th’ country, for you’ll be rotten
FTLN 1309
ere you be half ripe, and that’s the right virtue of
FTLN 1310
the medlar.""")

Retrieving documents for query: 'Then it will be
FTLN 1308 120
the earliest fruit i’ th’ country, for you’ll be rotten
FTLN 1309
ere you be half ripe, and that’s the right virtue of
FTLN 1310
the medlar.'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.27it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_e4535fbe_118',
  'content': 'the earliest fruit i’ th’ country, for you’ll be rotten\n ere you be half ripe, and that’s the right virtue of\n the medlar.\n  You have said, but whether wisely or no,\n let the forest judge.\nEnter Celia, as Aliena,  with a writing.\n, as Ganymede   Peace. Here comes my sister\n reading. Stand aside.\n, as Aliena, reads  \n Why should this a  desert be?\n  For it is unpeopled? No.\n Tongues I’ll hang on every tree\n  That shall civil sayings show.\n Some how brief the life of man\n  Runs his erring pilgrimage,\nFTLN 1287\nFTLN 1288 100\nFTLN 1289\nFTLN 1290\nFTLN 1291\nFTLN 1292\nFTLN 1293 105\nFTLN 1294\nFTLN 1295\nFTLN 1296\nFTLN 1297\nFTLN 1298 110\nFTLN 1299\nFTLN 1300\nFTLN 1301\nFTLN 1302\nFTLN 1303 115\nFTLN 1304\nFTLN 1305\nFTLN 1306\nFTLN 1307\nFTLN 1308 120\nFTLN 1309\nFTLN 1310\nFTLN 1311\nFTLN 1312\nFTLN 1313 125\nFTLN 1314\nFTLN 1315\nFTLN 1316\nFTLN 1317\nFTLN 1318 130\nFTLN 1319\nFTLN 1320',
  'metadata': {'doc_index': 118,
   

RAG Pipeline- VectorDB To LLM Output Generation

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [27]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage, SystemMessage

In [35]:
class OpenAILLM:
    def __init__(self, model_name: str = "gpt-6-luna", api_key: str =None):
        """
        Initialize OpenAI LLM
        
        Args:
            model_name: OpenAI model name (gpt-6-luna)
            api_key: OpenAI API key (or set OPENAI_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("OPENAI_API_KEY")
        
        if not self.api_key:
            raise ValueError("OpenAI API key is required. Set OPENAI_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatOpenAI(
            api_key=self.api_key,
            model_name=self.model_name,
            max_tokens=1024
        )
        
        print(f"Initialized OpenAI LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [36]:
# Initialize OpenAI LLM (you'll need to set OPENAI_API_KEY environment variable)
try:
    openai_llm = OpenAILLM(api_key=os.getenv("OPENAI_API_KEY"))
    print("OpenAI LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your OPENAI_API_KEY environment variable to use the LLM.")
    openai_llm = None

Initialized OpenAI LLM with model: gpt-6-luna
OpenAI LLM initialized successfully!


In [37]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("This is the very false gallop of verses")

Retrieving documents for query: 'This is the very false gallop of verses'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.12it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_416e989a_810',
  'content': 'From most true wretchedness. So thou, Posthumus,\n Wilt lay the leaven on all proper men;\n Goodly and gallant shall be false and perjured\n From thy great fail.—Come, fellow, be thou honest;\n Do thou thy master’s bidding. When thou seest him,\n A little witness my obedience. Look,\n I draw the sword myself.\n Take it, and hit\n The innocent mansion of my love, my heart.\n Fear not; ’tis empty of all things but grief.\n Thy master is not there, who was indeed\n The riches of it. Do his bidding; strike.\n Thou mayst be valiant in a better cause,\n But now thou seem’st a coward.\n, throwing down the sword    Hence, vile\n instrument!\n Thou shalt not damn my hand.\n   Why, I must die,\nFTLN 1710\nFTLN 1711 50\nFTLN 1712\nFTLN 1713\nFTLN 1714\nFTLN 1715\nFTLN 1716 55\nFTLN 1717\nFTLN 1718\nFTLN 1719\nFTLN 1720\nFTLN 1721 60\nFTLN 1722\nFTLN 1723\nFTLN 1724\nFTLN 1725\nFTLN 1726 65\nFTLN 1727\nFTLN 1728\nFTLN 1729\nFTLN 1730\nFTLN 1731 70\nFTLN 17

Integration Vectordb Context pipeline With LLM output

In [60]:
### Simple RAG pipeline with OpenAI LLM
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the OpenAI LLM (set your OPENAI_API_KEY in environment)
openai_api_key = os.getenv("OPENAI_API_KEY")

llm=ChatOpenAI(api_key=openai_api_key,model_name="gpt-6-luna",max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using OpenAI LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [61]:
answer=rag_simple("As You Like It",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'As You Like It'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.77it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.
